# Deployment Context & Drift Integration

Closes the loop: modeling → decisions → monitoring. How the system works after deployment.

Objective: describe operational lifecycle, drift signals, governance, failure modes. Not infrastructure—this project provides the model and policy, external systems handle serving.

Focus: integration with drift detection for continuous monitoring.

## Step 1: Deployment Context

How the model works in production. Key separation: model inference (pure prediction) vs policy enforcement (business logic).

In [ ]:
print("=== DEPLOYMENT CONTEXT ===\n")

print("""
**Production Workflow**:

1. **Data Ingestion**
   - New loan applications arrive (batch or streaming)
   - Feature extraction from application data and credit history
   - Feature validation and missingness checks

2. **Model Inference**
   - Input: Approved feature set (from Step 4, without PAY_0)
   - Model: Baseline Logistic Regression or LightGBM (selected based on governance criteria)
   - Output: Probability of Default (PD) for each applicant
   - Location: Model serving layer (API or batch job)

3. **Policy Enforcement**
   - Input: PD estimate, EAD (credit limit), LGD (business assumption)
   - Decision rule: Approve if PD × EAD × LGD < opportunity_cost
   - Output: Approval/rejection decision
   - Location: Decision engine (separate from model inference)

4. **Decision Logging**
   - Store: Applicant ID, features, PD, decision, timestamp
   - Purpose: Auditability, drift detection, model validation

**Architecture Separation**:
- Model inference: Pure prediction (PD estimation)
- Policy enforcement: Business logic (decision rule)
- Monitoring: Drift detection system (separate service)

**Scoring Mode**:
- **Batch scoring**: Daily/weekly batch of applications
- **Near-real-time**: API-based scoring for individual applications
- **Hybrid**: Batch for bulk, real-time for urgent cases

**Key Principle**: Model outputs PD; business logic makes decisions.
""")

## Step 2: Drift Signals

Define what is monitored and why each signal matters for operational decision-making.

In [ ]:
print("=== DRIFT SIGNALS ===\n")

print("""
**1. PD Distribution Drift**

**What**: Compare distribution of predicted PDs in production vs training/reference period

**Metrics**:
- Population Stability Index (PSI) on PD distribution
- Mean PD shift
- PD percentile shifts (e.g., 90th percentile)

**Why it matters**:
- Indicates model calibration drift
- May signal distribution shift in underlying population
- If PD distribution shifts upward → more rejections (policy impact)
- If PD distribution shifts downward → more approvals (risk impact)

**Threshold**: PSI > 0.25 indicates significant drift (industry standard)

**Action trigger**: Calibration check, potential threshold adjustment, or retraining


**2. Feature Distribution Drift**

**What**: Compare feature distributions in production vs training period

**Metrics**:
- PSI for each feature (or feature groups)
- Mean/median shifts for continuous features
- Category frequency shifts for categorical features

**Why it matters**:
- Indicates population shift (demographic changes, economic cycles)
- May explain PD distribution drift
- Features with high PSI may need investigation
- Critical for understanding *why* model performance changes

**Threshold**: PSI > 0.25 per feature (or > 0.10 for critical features)

**Action trigger**: Feature engineering review, model retraining if drift is persistent


**3. Decision-Level Drift**

**What**: Monitor actual business outcomes and decisions

**Metrics**:
- Approval rate (overall and by group)
- Expected loss (total and per approved loan)
- Default rate among approved loans
- Default rate by demographic group

**Why it matters**:
- Directly measures business impact
- Approval rate drift → policy may need adjustment
- Expected loss drift → financial risk may be increasing
- Default rate drift → model may be miscalibrated or population changed

**Threshold**: 
- Approval rate change > 5 percentage points
- Expected loss change > 10%
- Default rate change > 2 percentage points

**Action trigger**: Policy review, threshold adjustment, or model retraining


**4. Group-Level Drift (Fairness Monitoring)**

**What**: Monitor approval rates and outcomes by demographic groups

**Metrics**:
- Approval rate by group (sex, age, education, credit limit)
- Expected loss by group
- Default rate by group

**Why it matters**:
- Ensures fairness constraints are maintained (from Step 9)
- Detects disparate impact over time
- May indicate model bias or population shift

**Threshold**: 
- Approval rate disparity > 20 percentage points (80% rule violation)
- Group-level default rate > 2x overall default rate

**Action trigger**: Fairness audit, model review, or constraint enforcement


**5. Model Performance Drift**

**What**: Compare model discrimination and calibration over time

**Metrics**:
- Brier score (calibration)
- AUC (discrimination) - if labels available
- Calibration curve shifts

**Why it matters**:
- Indicates model degradation
- May require retraining or calibration adjustment

**Threshold**: 
- Brier score increase > 0.01
- AUC decrease > 0.05

**Action trigger**: Model retraining or calibration update
""")

## Step 3: Drift → Action Mapping

Explicitly define responses to drift signals. **Key principle**: Not all drift requires retraining.

In [ ]:
print("=== DRIFT → ACTION MAPPING ===\n")

print("""
**Decision Tree for Drift Response**:

┌─────────────────────────────────────────────────────────────┐
│ 1. PD DISTRIBUTION DRIFT (PSI > 0.25)                        │
└─────────────────────────────────────────────────────────────┘
│
├─→ Check feature distribution drift
│   │
│   ├─→ If feature drift is HIGH (PSI > 0.25 for key features)
│   │   → **RETRAIN MODEL** (population shift detected)
│   │
│   └─→ If feature drift is LOW (PSI < 0.10)
│       → **RECALIBRATE MODEL** (calibration drift only)
│       → Apply post-hoc calibration (Platt scaling or isotonic)
│
└─→ Check decision-level drift
    │
    ├─→ If approval rate changed significantly (>5pp)
    │   → **REVIEW THRESHOLD** (may need adjustment)
    │   → Recalculate threshold from updated cost assumptions
    │
    └─→ If expected loss changed significantly (>10%)
        → **TRIGGER HUMAN REVIEW** (business impact)
        → Assess if threshold adjustment is needed


┌─────────────────────────────────────────────────────────────┐
│ 2. FEATURE DISTRIBUTION DRIFT (PSI > 0.25 for key features) │
└─────────────────────────────────────────────────────────────┘
│
├─→ Investigate root cause
│   │
│   ├─→ Economic cycle shift (recession, boom)
│   │   → **RETRAIN MODEL** (fundamental population change)
│   │
│   ├─→ Demographic shift (new market, policy change)
│   │   → **RETRAIN MODEL** (population composition change)
│   │
│   └─→ Data quality issue (missingness, encoding change)
│       → **FIX DATA PIPELINE** (not a model issue)
│       → No retraining needed if data issue is resolved


┌─────────────────────────────────────────────────────────────┐
│ 3. DECISION-LEVEL DRIFT                                     │
└─────────────────────────────────────────────────────────────┘
│
├─→ Approval rate drift (>5 percentage points)
│   │
│   ├─→ If PD distribution also drifted
│   │   → **RETRAIN MODEL** (model-driven drift)
│   │
│   └─→ If PD distribution stable
│       → **REVIEW THRESHOLD** (policy may need adjustment)
│       → Check if business assumptions (LGD, opportunity cost) changed
│
├─→ Expected loss drift (>10%)
│   │
│   ├─→ If default rate increased
│   │   → **TRIGGER HUMAN REVIEW** (financial risk)
│   │   → Consider threshold adjustment or model retraining
│   │
│   └─→ If default rate stable
│       → **INVESTIGATE** (may be volume or EAD change)
│
└─→ Default rate drift (>2 percentage points)
    │
    ├─→ If model predicted this (high PD for defaults)
    │   → **ACCEPTABLE** (model working as expected)
    │
    └─→ If model did not predict this (low PD for defaults)
        → **RETRAIN MODEL** (model degradation)


┌─────────────────────────────────────────────────────────────┐
│ 4. GROUP-LEVEL DRIFT (Fairness Violation)                   │
└─────────────────────────────────────────────────────────────┘
│
├─→ Approval rate disparity > 20 percentage points
│   │
│   ├─→ If justified by risk differences (PD differences)
│   │   → **DOCUMENT** (policy-driven, acceptable)
│   │
│   └─→ If NOT justified by risk differences
│       → **ENFORCE FAIRNESS CONSTRAINT** (adjust threshold by group)
│       → **TRIGGER HUMAN REVIEW** (potential discrimination)
│
└─→ Group-level default rate > 2x overall default rate
    │
    └─→ **INVESTIGATE MODEL BIAS** (may need retraining with fairness constraints)


┌─────────────────────────────────────────────────────────────┐
│ 5. MODEL PERFORMANCE DRIFT                                  │
└─────────────────────────────────────────────────────────────┘
│
├─→ Brier score increase > 0.01 (calibration degradation)
│   │
│   └─→ **RECALIBRATE MODEL** (post-hoc calibration)
│       → If recalibration fails → **RETRAIN MODEL**
│
└─→ AUC decrease > 0.05 (discrimination degradation)
    │
    └─→ **RETRAIN MODEL** (model no longer captures patterns)


**Key Principles**:

1. **Not all drift requires retraining**:
   - Calibration drift → Recalibrate
   - Policy drift → Adjust threshold
   - Data quality drift → Fix pipeline

2. **Retrain only when**:
   - Population fundamentally changed (feature drift + PD drift)
   - Model performance degraded (AUC/Brier degradation)
   - Persistent drift that recalibration cannot fix

3. **Threshold adjustment when**:
   - Business assumptions changed (LGD, opportunity cost)
   - Approval rate drift without model drift
   - Fairness constraints require group-specific thresholds

4. **Human review when**:
   - Financial risk increased significantly
   - Fairness violations detected
   - Multiple drift signals simultaneously
   - Edge cases or model disagreement
""")

In [ ]:
print("=== GOVERNANCE & CONTROLS ===\n")

print("""
**1. Model Versioning**

**Versioning Scheme**:
- Format: `v{MAJOR}.{MINOR}.{PATCH}`
- MAJOR: Model architecture change (e.g., logistic → LightGBM)
- MINOR: Retraining with same architecture
- PATCH: Calibration update only

**Version Metadata**:
- Training date
- Training data period
- Feature set version
- Hyperparameters
- Performance metrics (Brier, AUC, calibration curve)
- Fairness metrics (group-level approval rates)

**Deployment Process**:
- New model version → A/B test against current version
- Monitor for 2-4 weeks before full rollout
- Rollback plan if new version underperforms


**2. Policy Versioning**

**Policy Components**:
- PD threshold (derived from LGD, opportunity cost)
- EAD calculation method
- LGD assumption
- Opportunity cost assumption
- Fairness constraints (group-level thresholds if applicable)

**Versioning**:
- Policy version tracked separately from model version
- Policy changes require business approval
- Document rationale for threshold changes

**Policy Update Triggers**:
- Business assumption changes (LGD, opportunity cost)
- Regulatory requirements
- Fairness constraint enforcement
- Risk appetite changes


**3. Auditability**

**Required Logs**:
- **Input features**: All features used for PD prediction
- **PD estimate**: Model output probability
- **Decision**: Approval/rejection
- **Decision rule**: Threshold and business assumptions used
- **Timestamp**: When decision was made
- **Model version**: Which model version was used
- **Policy version**: Which policy version was used

**Audit Trail Purpose**:
- Regulatory compliance (ECOA, FCRA adverse action explanations)
- Model validation and backtesting
- Dispute resolution
- Fairness audits
- Drift detection and analysis

**Retention**: Minimum 7 years (regulatory requirement)


**4. Interaction with Fairness Constraints**

**Fairness Monitoring** (from Step 9):
- Real-time tracking of approval rates by group
- Weekly reports on group-level outcomes
- Quarterly fairness audits

**Fairness Constraints**:
- Maximum approval rate disparity: 80% rule (min approval rate / max approval rate >= 0.8)
- Minimum approval rate per group: Set based on business policy
- Group-level default rate limits: Prevent excessive risk concentration

**Enforcement**:
- If fairness violation detected → Alert human reviewer
- Option 1: Adjust threshold by group (if risk differences justify)
- Option 2: Retrain model with fairness constraints
- Option 3: Manual review for edge cases

**Documentation**:
- Document all fairness violations and responses
- Regular reports to compliance team
- Annual fairness audit report


**5. Model Governance Workflow**

**Regular Reviews**:
- **Daily**: Automated drift monitoring (PSI, approval rates)
- **Weekly**: Review drift reports, expected loss trends
- **Monthly**: Model performance review, calibration checks
- **Quarterly**: Fairness audit, comprehensive model validation
- **Annually**: Full model validation, regulatory compliance review

**Change Management**:
- Model changes require approval from:
  - Data science team (technical validation)
  - Risk management (business validation)
  - Compliance (regulatory validation)
- Policy changes require approval from:
  - Business stakeholders
  - Risk management
  - Legal/compliance

**Rollback Procedures**:
- Previous model version always available
- Policy rollback to previous threshold
- Emergency stop: Suspend automated decisions, require manual review
""")

## Step 5: Failure Modes & Limitations

Explicitly list known blind spots, what the system cannot safely decide, and where manual intervention is required.

In [ ]:
print("=== FAILURE MODES & LIMITATIONS ===\n")

print("""
**1. Known Blind Spots**

**Temporal Limitations**:
- Model trained on historical data (may not capture future economic shocks)
- No explicit handling of economic cycles (recession, boom)
- Assumes stable relationship between features and default

**Feature Limitations**:
- Missing features: Income, employment status, debt-to-income ratio
- Proxy features (credit limit, education) may not capture true risk
- No external data: Macroeconomic indicators, industry trends

**Label Limitations**:
- Default definition is binary (month 0 default only)
- No handling of delayed defaults or censoring
- Label noise in edge cases (borderline repayment behavior)

**Population Limitations**:
- Trained on UCI dataset (may not represent current population)
- No explicit handling of new customer segments
- Assumes similar risk patterns across all applicants


**2. What This System Cannot Safely Decide**

**Edge Cases Requiring Manual Review**:
- **Extreme PD values**: PD > 0.5 or PD < 0.01 (model uncertainty)
- **Missing critical features**: If key features are missing (e.g., payment history)
- **Model disagreement**: If baseline and LightGBM disagree significantly
- **Fairness violations**: If group-level approval rate violates 80% rule
- **High-stakes applications**: Large loan amounts (EAD > threshold)
- **Unusual feature combinations**: Patterns not seen in training data

**Automatic Rejection Triggers**:
- PD > 0.5 (high risk, regardless of threshold)
- Missing critical features (data quality issue)
- Feature values outside training distribution (outliers)

**Automatic Approval Triggers**:
- None (all decisions go through policy rule)


**3. Where Manual Intervention is Required**

**Immediate Human Review**:
- Fairness violations (approval rate disparity > 20 percentage points)
- Expected loss spike (>20% increase in short period)
- Model performance degradation (AUC drop > 0.1)
- Multiple drift signals simultaneously
- Regulatory complaints or disputes

**Scheduled Human Review**:
- Weekly drift reports (review trends, not individual decisions)
- Monthly model performance review
- Quarterly fairness audits
- Annual model validation

**Escalation Criteria**:
- Financial risk: Expected loss exceeds portfolio limits
- Fairness risk: Persistent group-level disparities
- Model risk: Performance degradation not resolved by recalibration
- Operational risk: System errors, data quality issues


**4. System Limitations**

**Cannot Handle**:
- **Novel fraud patterns**: Model trained on historical data, may miss new fraud
- **Economic shocks**: Sudden economic changes not captured in training
- **Regulatory changes**: New regulations may require model updates
- **New product types**: Model trained on credit cards, may not generalize

**Assumptions**:
- Stable feature distributions (monitored via PSI)
- Stable default patterns (monitored via default rate)
- Stable business environment (monitored via approval rate, expected loss)

**Risk Mitigation**:
- Continuous monitoring (drift detection)
- Regular retraining (quarterly or when drift detected)
- Human oversight (review edge cases and violations)
- Model governance (versioning, auditability, rollback)


**5. Operational Safeguards**

**Circuit Breakers**:
- If approval rate changes > 10 percentage points in 24 hours → Alert + review
- If expected loss increases > 20% in 1 week → Alert + review
- If model performance degrades (AUC < 0.6) → Suspend automated decisions
- If fairness violation persists > 1 week → Suspend automated decisions for affected group

**Data Quality Checks**:
- Missingness thresholds: Alert if >5% missing for critical features
- Outlier detection: Flag values outside 3 standard deviations
- Feature stability: Alert if feature distribution changes significantly

**Model Health Checks**:
- Daily: PD distribution check (PSI)
- Weekly: Model performance check (if labels available)
- Monthly: Calibration check (Brier score)
- Quarterly: Full model validation


**6. Integration with Drift Detection System**

**This Project Provides**:
- Model (PD estimation)
- Policy (decision rule)
- Fairness constraints
- Governance framework

**Drift Detection System Provides**:
- PSI calculation and monitoring
- Feature distribution tracking
- Decision-level monitoring
- Alerting and reporting

**Integration Points**:
- Model outputs PD → Drift system monitors PD distribution
- Policy makes decisions → Drift system monitors approval rates
- Fairness constraints enforced → Drift system monitors group-level outcomes
- Drift detected → Triggers actions defined in this notebook

**Workflow**:
1. Model predicts PD for new applications
2. Policy applies decision rule
3. Decisions logged to drift detection system
4. Drift system monitors signals (PSI, approval rates, etc.)
5. If drift detected → Trigger action (recalibrate, retrain, adjust threshold)
6. Actions logged and reviewed
""")

## Summary

**Deployment Context & Drift Integration completed**:

1. **Deployment context**: Defined production workflow, separating model inference from policy enforcement

2. **Drift signals**: Identified 5 key monitoring signals (PD distribution, feature distribution, decision-level, group-level, model performance)

3. **Drift → action mapping**: Created decision tree for responding to drift (recalibrate, retrain, adjust threshold, human review)

4. **Governance & controls**: Documented versioning, auditability, fairness constraints, and governance workflow

5. **Failure modes & limitations**: Explicitly listed blind spots, edge cases, and operational safeguards

**Key principles**:
- Not all drift requires retraining (calibration vs population shift)
- Model outputs PD; business logic makes decisions
- Continuous monitoring enables proactive response
- Human oversight required for edge cases and violations

**Integration**:
- This project provides model, policy, and governance framework
- Drift detection system provides monitoring, alerting, and reporting
- Together, they form a complete operational lifecycle

**Project Status**: ✅ **COMPLETE**

All 10 steps of the credit default risk modeling lifecycle are now documented:
1. Problem & Decision Definition
2. Temporal Framing
3. Label Construction & Noise Analysis
4. Data Audit & Leakage Defense
5. Baseline Decision Model
6. Thresholding & Policy Design
7. Data-Centric Iteration
8. Higher-Capacity Model
9. Fairness, Stability & Stress Testing
10. Deployment Context & Drift Integration